# 09. E-commerce Benchmark Generator

Notebook sinh benchmark tùy chỉnh cho agentic RAG e-commerce.
- Hỗ trợ: single spec, lines, lines + specs, top-N, combined OR, ambiguous, compare, multi-turn, hard.
- Truy vấn Supabase để lấy ground truth chính xác.
- Xoay vòng API key Gemini + rate-limit.
- Output: JSONL chuẩn RAGAS-friendly / benchmark.

In [9]:
import sys
from pathlib import Path

# Đảm bảo import được từ rag-service/
repo_root = Path.cwd().parent.resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.h_evaluation.ecommerce_benchmark_generator import (
    EcommerceBenchmarkGenerator,
    ProductCatalog,
    GeminiLLM,
)
import json

print('OK')

OK


## 1. Kiểm tra catalog

In [10]:
catalog = ProductCatalog()
print(f'Loaded {len(catalog.df)} products')
print('Brands:', catalog.distinct_brands()[:10])
print('Laptop lines:', catalog.distinct_lines(category='laptop')[:10])

Loaded 1000 products
Brands: ['ASUS', 'Acer', 'Apple', 'Dell', 'Gigabyte', 'HONOR', 'HP', 'Huawei', 'Hãng khác', 'INOI']
Laptop lines: ['Apple Mac Studio', 'Apple MacBook Air', 'Apple Studio Display 27 5K Chân Đế Cố Định', 'Apple Studio Display 27 5K Chân Đế Điều Chỉnh', 'Apple Studio Display 27 5K Chân Đế Điều Chỉnh Màn Nano', 'Apple Studio Display 27 5K Ngàm VESA', 'Apple Studio Display XDR 27 5K Chân Đế Điều Chỉnh', 'Apple Studio Display XDR 27 5K Ngàm VESA Màn Nano', 'Laptop ASUS', 'Laptop ASUS ExpertBook']


## 2. Khởi tạo generator

In [ ]:
USER_TOKEN = "eyJhbGciOiJFUzI1NiIsImtpZCI6ImNiZDkwZGZjLTFkMmEtNDE5My1iNzE2LTlkMDgxOGM2MGEyNCIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL3JpeGNlbm5kc3JscGl2emJjbm1mLnN1cGFiYXNlLmNvL2F1dGgvdjEiLCJzdWIiOiJmZjY0MWYyNi0zYWRhLTQ3ZDAtOWJmMi0xZjRiNzE2NTQwNjQiLCJhdWQiOiJhdXRoZW50aWNhdGVkIiwiZXhwIjoxNzg1OTUyOTYyLCJpYXQiOjE3ODU4NjY1NjIsImVtYWlsIjoidnVnaWFraGFpMjAwNEBnbWFpbC5jb20iLCJwaG9uZSI6IiIsImFwcF9tZXRhZGF0YSI6eyJwcm92aWRlciI6Imdvb2dsZSIsInByb3ZpZGVycyI6WyJnb29nbGUiXX0sInVzZXJfbWV0YWRhdGEiOnsiYWRkcmVzcyI6IiIsImF2YXRhcl91cmwiOiJodHRwczovL2xoMy5nb29nbGV1c2VyY29udGVudC5jb20vYS9BQ2c4b2NKdW9idmozc1ZPZDRsTE1JUVg2d3l4MEtncjJRNFZvZnlVM2pJYVhLN1p4LV9hcmc9czk2LWMiLCJiaXJ0aGRheSI6IjIwMDAtMDItMjAiLCJlbWFpbCI6InZ1Z2lha2hhaTIwMDRAZ21haWwuY29tIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsImZ1bGxfbmFtZSI6IkIyMkRDS0gwNjVfVsWpIEdpYSBLaOG6o2kiLCJpc3MiOiJodHRwczovL2FjY291bnRzLmdvb2dsZS5jb20iLCJuYW1lIjoiQjIyRENLSDA2NV9WxakgR2lhIEto4bqjaSIsInBob25lIjoiIiwicGhvbmVfdmVyaWZpZWQiOmZhbHNlLCJwaWN0dXJlIjoiaHR0cHM6Ly9saDMuZ29vZ2xldXNlcmNvbnRlbnQuY29tL2EvQUNnOG9jSnVvYnZqM3NWT2Q0bExNSVFYNnd5eDBLZ3IyUTRWb2Z5VTNqSWFYSzdaeC1fYXJnPXM5Ni1jIiwicHJvdmlkZXJfaWQiOiIxMDUxNTE0NjExMTM0OTM3ODM1MzUifSwicm9sZSI6ImF1dGhlbnRpY2F0ZWQiLCJhYWwxIjoiYWFsMSIsImFsciI6W3sibWV0aG9kIjoib2F1dGgiLCJ0aW1lc3RhbXAiOjE3ODU3ODQ4NX1dLCJzZXNzaW9uX2lkIjoiZjU0ZDExM2QtZDhkOC00Mjk1LWI2ZjctZDM5YTgwZDFhYmE5IiwiaXNfYW5vbnltb3VzIjpmYWxzZX0.bAk8jC1B0pxcaM1oBu_OGY2xEOINjtTjg-qOZCSe8IRcke0AwSEpAYIA4SKAlYAfqqtbb8CWyUK3zIWZ649fLQ"
USER_ID = "ff641f26-3ada-47d0-9bf2-1f4b71654064"

gen = EcommerceBenchmarkGenerator(
    catalog=catalog,
    llm=GeminiLLM(min_interval_s=5.0),
    user_token=USER_TOKEN,
    current_user_id=USER_ID,
)
print('Generator ready')
print('Fetched user orders:', len(gen._fetch_user_orders()))

## 3. Sinh benchmark đầy đủ usecase (20 mẫu mỗi loại)

### 📝 Chi tiết cấu hình đầu vào và cấu trúc dữ liệu đầu ra

#### 1. Các tham số cấu hình đầu vào (`gen.generate_all`)

* **`counts` (Số lượng câu hỏi theo nhóm)**:
  * `single_spec`: Câu hỏi về 1 thông số kỹ thuật của 1 sản phẩm cụ thể (RAM, CPU, Pin...).
  * `lines`: Liệt kê các dòng sản phẩm của hãng X trong phân khúc giá cụ thể.
  * `lines_specs`: Liệt kê dòng sản phẩm kèm thông số kỹ thuật chi tiết của từng dòng.
  * `top_n`: Đề xuất N sản phẩm tốt nhất/đáng mua nhất của hãng X trong phân khúc giá.
  * `combined_or`: Câu hỏi chứa điều kiện ghép "HOẶC" (ví dụ: MacBook Pro hoặc MacBook Air).
  * `compare`: Yêu cầu so sánh tính năng/thông số giữa 2-3 sản phẩm cụ thể.
  * `ambiguous`: Câu hỏi mơ hồ thiếu bối cảnh để kiểm tra khả năng hỏi làm rõ thông tin của Agent.
  * `multi_turn`: Kịch bản hội thoại nhiều lượt nối tiếp ngữ cảnh.
  * `hard`: Tình huống nghiệp vụ phức tạp (ví dụ: đổi trả hàng, áp dụng mã ưu đãi sinh viên).
  * `order_account`: Tra cứu/hủy đơn hàng.
  * `risk_ticket`: Khiếu nại/đòi hoàn tiền.
  * `attack`: Prompt injection / yêu cầu vi phạm chính sách.
  * `compound`: Kết hợp thông tin sản phẩm + chính sách/ưu đãi.
* **`enable_multi_turn`**: `True` / `False` để bật/tắt sinh kịch bản hội thoại nhiều lượt.
* **`extra_samples`**: Danh sách câu hỏi mẫu tự định nghĩa. LLM tự động phân tích và sinh các câu tương tự dựa trên dữ liệu sản phẩm thật từ database.

---

#### 2. Cấu trúc dữ liệu đầu ra (Output JSONL Fields)

Mỗi bản ghi trong file `.jsonl` chứa:

* **`id`**: Mã định danh duy nhất (`category_timestamp_counter`).
* **`category`**: Phân loại chủ đề câu hỏi.
* **`question`**: Câu hỏi tiếng Việt tự nhiên gửi vào Agent.
* **`expected_tool_calls`**: Tool và tham số mong đợi Agent gọi (theo đúng schema `product_search`, `product_compare`, `policy_search`, `order_lookup`).
* **`ground_truth`**: Đáp án chuẩn từ database:
  * `product_ids`: ID sản phẩm đúng.
  * `answer_summary`: Câu trả lời mẫu (tên + giá + thông số).
  * `specs` / `specs_per_line`: Thông số chi tiết.
* **`metadata`**: Siêu dữ liệu lọc (hãng, khoảng giá, scenario, persona...).
* **`contexts`**: Mô tả gốc sản phẩm để RAGAS tính `Faithfulness`.
* **`turns`** *(chỉ `multi_turn`)*: Danh sách các lượt hội thoại, mỗi lượt có `question`, `expected_tool_calls`, `ground_truth`.

In [ ]:
%%time
records, diversity_report = gen.generate_all(
    counts={
        'single_spec': 20,
        'lines': 20,
        'lines_specs': 20,
        'top_n': 20,
        'combined_or': 20,
        'compare': 20,
        'ambiguous': 20,
        'multi_turn': 20,
        'hard': 20,
        'order_account': 20,
        'risk_ticket': 20,
        'attack': 20,
        'compound': 20,
    },
    enable_multi_turn=True,
)
print(f'Generated {len(records)} records')
print('Diversity report:')
for k, v in sorted(diversity_report.items()):
    print(k, v)

## 4. Kiểm tra mẫu đầu ra

In [ ]:
import os

target_path = repo_root / "src" / "h_evaluation" / "test_sets" / "ecommerce_benchmark_20each_new.jsonl"

# Thực hiện lưu file
gen.save(records, path=str(target_path))
print("Saved to", target_path)

# Kiểm tra sự tồn tại của file
print("File exists:", target_path.exists())

# 7. Kiểm chứng dữ liệu đã lưu
from collections import Counter

with open(target_path, 'r', encoding='utf-8') as f:
    saved = [json.loads(line) for line in f if line.strip()]

pid_set = {p['id'] for p in catalog.df}
valid = 0
for r in saved:
    if 'turns' in r:
        ok = all(all(pid in pid_set for pid in t['ground_truth'].get('product_ids', [])) for t in r['turns'])
    else:
        ok = all(pid in pid_set for pid in r.get('ground_truth', {}).get('product_ids', []))
    if ok:
        valid += 1

print(f'File: {target_path}')
print(f'Tổng records: {len(saved)}')
print('Phân bố:', dict(Counter(r['category'] for r in saved)))
print(f'Valid product ids: {valid}/{len(saved)}')

In [14]:
def quick_validate(record, catalog):
    """Kiểm tra product_ids trong ground_truth có tồn tại không."""
    if 'turns' in record:
        return True
    pids = record.get('ground_truth', {}).get('product_ids', [])
    if not pids:
        return record['category'] in ('ambiguous', 'hard')
    found = sum(1 for p in catalog.df if p['id'] in pids)
    return found == len(pids)

valid = [quick_validate(r, catalog) for r in records]
print('Valid records:', sum(valid), '/', len(valid))
for i, ok in enumerate(valid):
    if not ok:
        print('Invalid:', records[i]['id'], records[i]['category'])

Valid records: 169 / 243
Invalid: lines_20260730154438_0031 lines
Invalid: lines_specs_20260730154458_0042 lines_specs
Invalid: lines_specs_20260730154458_0048 lines_specs
Invalid: lines_specs_20260730154458_0053 lines_specs
Invalid: top_n_20260730154518_0063 top_n
Invalid: top_n_20260730154518_0064 top_n
Invalid: combined_or_20260730154538_0081 combined_or
Invalid: combined_or_20260730154538_0082 combined_or
Invalid: combined_or_20260730154538_0083 combined_or
Invalid: combined_or_20260730154538_0085 combined_or
Invalid: combined_or_20260730154538_0088 combined_or
Invalid: combined_or_20260730154538_0092 combined_or
Invalid: combined_or_20260730154538_0096 combined_or
Invalid: combined_or_20260730154538_0097 combined_or
Invalid: order_account_20260730154803_0161 order_account
Invalid: order_account_20260730154808_0162 order_account
Invalid: order_account_20260730154813_0163 order_account
Invalid: order_account_20260730154817_0164 order_account
Invalid: order_account_20260730154823_016

## 6. Lưu benchmark

In [15]:
import os

target_path = repo_root / "src" / "h_evaluation" / "test_sets" / "ecommerce_benchmark_20each.jsonl"

# Thực hiện lưu file
output_path = gen.save(records, path=str(target_path))
print("Saved to", output_path)

# Kiểm tra sự tồn tại của file
print("File exists:", target_path.exists())

✅ Đã lưu 243 records tại D:\create\Agenttic-RAG-for-e-commerce\rag-service\src\h_evaluation\test_sets\ecommerce_benchmark_20each.jsonl
Saved to None
File exists: True


## 8. Ghi chú mở rộng

- Để thêm usecase mới: dùng `gen.generate_from_samples([sample_question], n_per_sample=3)`.
- Để tăng số lượng: chỉnh `counts`.
- File output có thể đưa vào đánh giá bằng RAGAS metrics.

In [16]:
# 7. Kiểm chứng dữ liệu đã lưu
from collections import Counter

with open(target_path, 'r') as f:
    saved = [json.loads(line) for line in f if line.strip()]

pid_set = {p['id'] for p in catalog.df}
valid = 0
for r in saved:
    if 'turns' in r:
        ok = all(all(pid in pid_set for pid in t['ground_truth'].get('product_ids', [])) for t in r['turns'])
    else:
        ok = all(pid in pid_set for pid in r.get('ground_truth', {}).get('product_ids', []))
    if ok:
        valid += 1

print(f'File: {target_path}')
print(f'Tổng records: {len(saved)}')
print('Phân bố:', dict(Counter(r['category'] for r in saved)))
print(f'Valid product ids: {valid}/{len(saved)}')

File: D:\create\Agenttic-RAG-for-e-commerce\rag-service\src\h_evaluation\test_sets\ecommerce_benchmark_20each.jsonl
Tổng records: 243
Phân bố: {'single_spec': 20, 'lines': 20, 'lines_specs': 20, 'top_n': 20, 'combined_or': 20, 'compare': 20, 'ambiguous': 20, 'multi_turn': 20, 'order_account': 20, 'risk_ticket': 20, 'attack': 20, 'compound': 20, 'laptop': 3}
Valid product ids: 243/243
